# Assignment 5: Autoencoders
Date: 

Report produced by:  
COMP 3705 Group 3

## Load the data and inspect the header

In [ ]:
import pandas as pd
import numpy as np
import math as m
import seaborn as sns
import matplotlib.pyplot as plt

# Load the dataset
df = pd.read_csv("")
df_clean = df.copy()
df_clean.head(10)

In [ ]:
df_clean.describe()

In [ ]:
cc_columns = df_clean.columns.to_list()
print(cc_columns)

## Observations 
### Meaning of the Columns


### Observations on Column Statistics


### Drop Unnecessary Columns

In [ ]:
print(df_clean["CUST_ID"].is_unique)
if df_clean["CUST_ID"].is_unique == True:
    df_clean = df_clean.drop(columns="CUST_ID")

### Check Dtypes, Missing Values, & Duplicate Rows

In [ ]:
df_clean.info()

### Data Type Observations


### Missing Value Handling

### Missing Value Analysis

### Check Duplicated Rows

In [ ]:
df_clean.duplicated(keep=False).sum()

### Findings and Cleaning

**1. Data Structure and Types**  

**2. Dropped Columns**

**3. Missing values**  

**4. Duplicate Data Entries**  

**5. Cleaning Steps**:

**Result:**  
 

## Data Visualizations

In [ ]:
plt.figure(figsize=(14, 10))

# Plot a heatmap of the features correlation
sns.heatmap(df_clean.corr(), annot=True, cmap="Oranges", center=0, linewidths=0.3)
plt.title("")
plt.show()

# Create a pairplot using a random sample of points (1500)
df_sample = df_clean.sample(n=1500, random_state=42)
sns.pairplot(df_sample,diag_kind="hist",plot_kws={"alpha": 0.6, "s": 20})
plt.suptitle("", y=1.01)
plt.show()

## Correlation Matrix & Pairplot Observations

**Strong positive correlations:**

**Negative correlations:**

**Weak relationships:**

**Histograms:**

**Scatter plots:**


## PCA

In [ ]:
from sklearn.preprocessing import MinMaxScaler as scaler
from sklearn.decomposition import PCA
def scale_df(df):
    df = scaler().fit_transform(df)
    return pd.DataFrame(df)

def apply_pca(df,n=0):
    if n == 0:
        pca_model = PCA(random_state=42)
    else:
        pca_model = PCA(n_components=n,random_state=42)
    data_transformed = pca_model.fit_transform(df.values)
    data_var_ratio = pca_model.explained_variance_ratio_*100
    data_var_ratio = np.cumsum(sorted(data_var_ratio,reverse=True))
    return pd.DataFrame(data_transformed), data_var_ratio, pca_model

df_scaled = scale_df(df_clean,)
data_t, data_var, pca_model = apply_pca(df_scaled)
print("Expplained Variance in Data:\n",data_var)


As we are wanting to retain at least 95% of the data's explained variance, we need to keep at least 7 components after PCA. Although the 7th component retains 96.54% of the data, which is greater than we need, reducing components to 6 brings the explained variance down to 94.72%. As we want a minimum of 95%, we will be proceeding with PCA using 7 components.

In [ ]:
data_t, data_var, pca_model = apply_pca(df_scaled,n=7)
print("Expplained Variance in Data:\n",data_var)
data_t.head(15)

In [ ]:
data_t.describe()

## Model Selection

In [ ]:
from sklearn.cluster import KMeans
from sklearn.cluster import AgglomerativeClustering as AC
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score as s_score, silhouette_samples as s_samples

def check_range(k_bounds):
    # Compare parameter types, change return based on input
    if type(k_bounds) == int or type(k_bounds) == float:
        if k_bounds < 2:
            k_bounds = [2]
        else:
            k_bounds = [round(k_bounds)]
    elif type(k_bounds) == tuple:
        k_bounds = list(k_bounds)
        k_bounds.sort()
        if k_bounds[0] < 2:
            k_bounds[0] = 2
    elif type(k_bounds) == list:
        k_bounds.sort()
        if k_bounds[0] < 2:
            k_bounds[0] = 2
    return k_bounds

def get_kmeans(x_var, y_var=None, k_range=2, lamda=0.0001, epoch=500):
    # Determine number k range. Default: 2
    num_clusters = check_range(k_range)

    # Initiallize containers and counters
    kmeans_metrics = dict()
    j = 0

    # Run K-Means for k number of clusters
    for k in range(num_clusters[0],num_clusters[-1]+1):
        if type(epoch) != list:
            if type(epoch) == int or type(epoch) == float:
                epoch = [round(epoch)]
            if type(epoch) == set or type(epoch) == tuple:
                epoch = list(epoch)
        for i in sorted(epoch):
            kmeans_model = KMeans(k, random_state=42, max_iter=i, tol=lamda).fit(x_var)
            # Add metrics to history
            centroids = kmeans_model.cluster_centers_
            labels = kmeans_model.labels_
            SSE = kmeans_model.inertia_
            silhouette_values = s_samples(x_var, labels)
            silhouette_avg = silhouette_values.mean()
            
            j += 1
        # Store model metrics in dictionary
            metrics = {
                "Algorithm": "KMeans",
                "Iteration": j,
                "Silhouette Score": silhouette_avg,
                "Silhouette Coefficients": silhouette_values,
                "SSE": SSE,
                "Clusters Found": len(centroids),
                "Centroids": centroids,
                "Labels": labels,
            }
            kmeans_metrics[f"Iteration {j}"] = metrics
    kmeans_df = pd.DataFrame(kmeans_metrics).transpose().drop(columns=["Labels", "Silhouette Coefficients", "Iteration", "Centroids"])
    return kmeans_metrics, kmeans_df

def get_agg_clustering(x_val, min_clusters=2, max_clusters=2, distance_metric=["euclidean"], linkage=["ward"], max_linkage_distance=[None]):
    # Create containers for results
    iteration = 0
    ac_metrics = dict()

    # Sets min_clusters to 1 if value passed is lower
    if min_clusters < 1:
        min_clusters = 1
    
    for num in range(min_clusters, max_clusters+1):
        for distance in distance_metric:
            for link in linkage:
                if link == "ward" and distance != "euclidean":
                    continue
                for threshold in max_linkage_distance:
                    if threshold != None:
                        if distance == "cosine":
                            continue
                        if num != min_clusters:
                            continue
                        model = AC(n_clusters=None, affinity=distance, linkage=link, distance_threshold=threshold, compute_full_tree=True).fit(x_val)
                    else:
                        model = AC(n_clusters=num, affinity=distance, linkage=link).fit(x_val)

                    iteration += 1
                    labels = model.labels_
                    silhouette_values = s_samples(x_val,labels,metric=distance)
                    metrics = {"Algorithm": "Hierarchical Clustering",
                               "n_clusters": "None" if threshold != None else num,
                               "distance_metric": distance,
                               "linkage": link,
                               "distance_threshold": threshold,
                               "Silhouette Score": silhouette_values.mean(),
                               "Silhouette Coefficients": silhouette_values,
                               "Labels": model.labels_,
                               "Clusters Found": model.n_clusters_,
                               "Merge History": model.children_,
                               "Iteration": iteration,
                            }
                    ac_metrics[f"Iteration {iteration}"] = metrics
    ac_results = pd.DataFrame(ac_metrics).transpose().drop(columns=["Labels", "Merge History", "Silhouette Coefficients","Iteration"])
    ac_results = ac_results[["Algorithm", "Clusters Found", "Silhouette Score", "n_clusters", "distance_metric", "linkage", "distance_threshold"]]
    return ac_metrics, ac_results

def get_dbscan(x_val, min_n=[5], eps_val=[0.5], distance_metric=["euclidean"], p_val=[2], max_clusters=None, model_param=None):
    i = 0
    e = 0
    db_metrics = dict()
    db_err = dict()
    # Allows modelling specific sets of parameters without testing every permutation.
    if model_param != None:
        for model_n in model_param:
            eps_val = model_param[model_n]["eps"]
            min_n = model_param[model_n]["min_n"]
            distance_metric = model_param[model_n]["metric"]
            p_val = model_param[model_n]["p_val"]
            for dist in distance_metric:
                for n in min_n:
                    for eps in eps_val:
                        if n <= 1 and eps == 0:
                            continue
                        for p in p_val:
                            if dist != "minkowski" and p != p_val[0]:
                                continue
                            model_n = DBSCAN(eps, min_samples=n,metric=dist, p=p).fit(x_val)                        
                            labels = model_n.labels_
                            n_clusters = max(model_n.labels_ + 1)
                            if (max_clusters != None and n_clusters > max_clusters) or n_clusters < 2:
                                e += 1
                                db_err[f"Iteration {e}"] = {
                                "Algorithm": "DBSCAN",
                                "Epsilon": eps,
                                "Min Samples": n,
                                "Distance Metric": dist,
                                "P Value": p,
                                "Noise (%)": np.sum(labels == -1)/len(labels)*100,
                                # "Silhouette Score": s_score(x_val,model_n.labels_,metric=dist),
                                # "Silhouette Coefficients": s_samples(x_val,model_n.labels_,metric=dist),
                                # "Labels": model_n.labels_,
                                "Clusters Found": n_clusters,
                                }
                                continue
                            i += 1
                            silhouette_values = s_samples(x_val,model_n.labels_,metric=dist)
                            model_metrics = {
                                "Algorithm": "DBSCAN",
                                "Iteration": i,
                                "Epsilon": eps,
                                "Min Samples": n,
                                "Distance Metric": dist,
                                "P Value": p,
                                "Noise (%)": np.sum(labels == -1)/len(labels)*100,
                                "Silhouette Score": silhouette_values.mean(),
                                "Silhouette Coefficients": silhouette_values,
                                "Labels": model_n.labels_,
                                "Clusters Found": n_clusters,
                            }
                            db_metrics[f"Iteration {i}"] = model_metrics
    else:
        for dist in distance_metric:
            for n in min_n:
                for eps in eps_val:
                    if n <= 1 and eps == 0:
                        continue
                    for p in p_val:
                        if dist != "minkowski" and p != p_val[0]:
                            continue
                        model_n = DBSCAN(eps, min_samples=n,metric=dist, p=p).fit(x_val)                        
                        labels = model_n.labels_
                        n_clusters = max(model_n.labels_ + 1)
                        if (max_clusters != None and n_clusters > max_clusters) or n_clusters < 2:
                            e += 1
                            db_err[f"Iteration {e}"] = {
                            "Algorithm": "DBSCAN",
                            "Epsilon": eps,
                            "Min Samples": n,
                            "Distance Metric": dist,
                            "P Value": p,
                            "Noise (%)": np.sum(labels == -1)/len(labels)*100,
                            # "Silhouette Score": s_score(x_val,model_n.labels_,metric=dist),
                            # "Silhouette Coefficients": s_samples(x_val,model_n.labels_,metric=dist),
                            # "Labels": model_n.labels_,
                            "Clusters Found": n_clusters,
                            }
                            continue
                        i += 1
                        silhouette_values = s_samples(x_val,model_n.labels_,metric=dist)
                        model_metrics = {
                            "Algorithm": "DBSCAN",
                            "Iteration": i,
                            "Epsilon": eps,
                            "Min Samples": n,
                            "Distance Metric": dist,
                            "P Value": p,
                            "Noise (%)": np.sum(labels == -1)/len(labels)*100,
                            "Silhouette Score": silhouette_values.mean(),
                            "Silhouette Coefficients": silhouette_values,
                            "Labels": model_n.labels_,
                            "Clusters Found": n_clusters,
                        }
                        db_metrics[f"Iteration {i}"] = model_metrics

    db_results = pd.DataFrame(db_metrics).transpose().drop(columns=["Labels", "Silhouette Coefficients", "Iteration"])
    db_results = db_results[["Algorithm","Clusters Found","Distance Metric","Min Samples","Epsilon","P Value","Noise (%)","Silhouette Score"]]
    db_err_results = pd.DataFrame(db_err).transpose()
    return db_metrics, db_results, db_err_results

def plot_silhouette(metrics):
    # Plot Silhouette vs Cluster
    model_num = metrics["Iteration"]
    model_type = metrics["Algorithm"]
    k = int(metrics["Clusters Found"])
    labels = metrics["Labels"]
    silhouette_avg = metrics["Silhouette Score"]
    silhouette_values = metrics["Silhouette Coefficients"]

    prev_cluster_size = 0
    silhouette_plot, axis = plt.subplots(figsize=(8,6))
    for i in range(0,k):
        tick_loc = []
        for n in range(0,k):
            nth_cluster_values = silhouette_values[labels==n]
            nth_cluster_values.sort()
            nth_cluster_size = len(nth_cluster_values)
            tick_loc.append(nth_cluster_size/2+prev_cluster_size)
            y_upper_bounds = prev_cluster_size + nth_cluster_size
            axis.fill_betweenx(np.arange(prev_cluster_size,y_upper_bounds),0,nth_cluster_values)
            prev_cluster_size += nth_cluster_size
        axis.set_yticks(tick_loc)
        axis.set_yticklabels(np.arange(0,k), fontsize=8)
        axis.axvline(silhouette_avg,c=(36/255.0, 9/255.0, 82/255.0),linestyle="--")
        axis .set_title(f"Silhouette Scores for Iteration {model_num} (Algrithm: {model_type}, Clusters: {k})")
        axis.set_ylabel("Cluster")
        axis.set_xlabel("Silhouette Coefficient")
        plt.show()

def plot_elbow_graph(metrics):
    first_iter = last_iter = list(metrics.keys())[0]
    last_iter = list(metrics.keys())[-1]
    n_cluster = (metrics[first_iter]["Clusters Found"], metrics[last_iter]["Clusters Found"])
    tot_iter = metrics[last_iter]["Iteration"]
    model_sse = []
    for iter in range(1,tot_iter+1):
        model_sse.append(metrics[f"Iteration {iter}"]["SSE"])
    plt.plot(range(n_cluster[0],n_cluster[1]+1),model_sse,'o-',)
    plt.title("SSE vs Number of Clusters")
    plt.xlabel("K-Value")
    plt.ylabel("SSE")
    plt.show()    

In [ ]:
def KMeans_breakdown(model_metrics,plot_elbow=False):
    breakdown = dict()
    for model in model_metrics:
        breakdown[model] = dict()
        plot_silhouette(model_metrics[model])
        n_clusters = model_metrics[f"{model}"]["Clusters Found"]
        breakdown[model][f"Num Neg Silhouette"] = 0
        for n in range(n_clusters):
            breakdown[model][f"Cluster {n}"] = np.sum(model_metrics[model]["Labels"] == n)
            breakdown[model][f"Num Neg Silhouette"] += sum(1 for x in model_metrics[model]["Silhouette Coefficients"] if x < 0)
        breakdown[model]["Algorithm"] = model_metrics[model]["Algorithm"]
        breakdown[model]["Silhouette Score"] = model_metrics[model]["Silhouette Score"]
        breakdown[model]["SSE"] = model_metrics[model]["SSE"]
    if len(model_metrics) > 1 and plot_elbow == True:
        plot_elbow_graph(model_metrics)
    df_kmeans = pd.DataFrame(breakdown).transpose()
    cluster_order = [f"Cluster {n}" for n in range(max(model_metrics[model]["Clusters Found"] for model in model_metrics))]
    df_kmeans.reindex(cluster_order)
    return df_kmeans

def agg_clust_breakdown(model_metrics):
    breakdown = dict()
    for model in model_metrics:
        breakdown[model] = dict()
        plot_silhouette(model_metrics[model])
        n_clusters = model_metrics[f"{model}"]["Clusters Found"]
        breakdown[model][f"Num Neg Silhouette"] = 0
        for n in range(n_clusters):
            breakdown[model][f"Cluster {n}"] = np.sum(model_metrics[model]["Labels"] == n)
            breakdown[model][f"Num Neg Silhouette"] += sum(1 for x in model_metrics[model]["Silhouette Coefficients"] if x < 0)
        breakdown[model]["Algorithm"] = model_metrics[model]["Algorithm"]
        breakdown[model]["Silhouette Score"] = model_metrics[model]["Silhouette Score"]
        breakdown[model]["Metric"] = model_metrics[model]["distance_metric"]
    df_aggcluster = pd.DataFrame(breakdown).transpose()
    cluster_order = [f"Cluster {n}" for n in range(max(model_metrics[model]["Clusters Found"] for model in model_metrics))]
    df_aggcluster.reindex(cluster_order)
    return df_aggcluster

def DBSCAN_breakdown(model_metrics):
    breakdown = dict()
    for model in model_metrics:
        breakdown[model] = dict()
        plot_silhouette(model_metrics[model])
        n_clusters = model_metrics[f"{model}"]["Clusters Found"]
        # print("Number of Clusters: ", n_clusters)
        breakdown[model] = {"Number of Clusters": n_clusters,
                            "Noise": np.sum(model_metrics[model]["Labels"] == -1)}
        breakdown[model][f"Num Neg Silhouette"] = 0
        for n in range(n_clusters):
            breakdown[model][f"Cluster {n}"] = np.sum(model_metrics[model]["Labels"] == n)
            breakdown[model][f"Num Neg Silhouette"] += sum(1 for x in model_metrics[model]["Silhouette Coefficients"] if x < 0)
        breakdown[model]["Algorithm"] = model_metrics[model]["Algorithm"]
        breakdown[model]["Silhouette Score"] = model_metrics[model]["Silhouette Score"]
        breakdown[model]["Metric"] = model_metrics[model]["Distance Metric"]
    df_dbscan = pd.DataFrame(breakdown).transpose()
    cluster_order = [f"Cluster {n}" for n in range(max(model_metrics[model]["Clusters Found"] for model in model_metrics))]
    df_dbscan.reindex(cluster_order)
    return df_dbscan

def model_breakdown(model,df_labeled):
    # Creates a dataframe for each cluster and assigns them to a variable
    clusters_df = dict()
    for i in range(min(model["Labels"]),model["Clusters Found"]):
        if i == -1:
            noise_df = df_labeled[df_labeled["Cluster"] == i]
        else:
            clusters_df[i] = df_labeled[df_labeled["Cluster"] == i]
        clusters_df[i].reset_index(drop=True, inplace=True)
    return clusters_df

### Final Selection

## Conclusion

In [ ]:
def label_data(data, chosen_model):
    labels = chosen_model["Labels"]
    # Add cluster labels to the features DataFrame
    df_labeled = data.copy()
    df_labeled["Cluster"] = labels
    return df_labeled
def restore_data(data,model):
    data_re = model.inverse_transform(data)
    return pd.DataFrame(data_re)

# best_model = 
data_labeled = label_data(data_t, best_model)
data_labeled.head(10)

### Cluster Visualizations

In [ ]:
# Plot Scatterplot of chosen features coloured by cluster

plt.figure(figsize=(10, 6))
sns.scatterplot(data=data_labeled, x=0,y=1,hue="Cluster",alpha=0.6,s=30)
plt.title("Principal Component: 0 vs 1")
plt.xlabel("Principal Component 0")
plt.ylabel("Principal Component 1")
plt.show()

### PCA Scatter Plot Observations


### Cluster Decomposition

In [ ]:
def plot_cluster_distribution(df_labeled):
    # Plot Histogram of chosen model cluster breakdown
    plt.figure(figsize=(7, 5))
    sns.countplot(data=df_labeled, x="Cluster", palette="tab10")
    plt.title(f"Iteration 1: Cluster Distribution")
    plt.xlabel("Cluster")
    plt.ylabel("Number of Customers")
    for i, count in df_labeled["Cluster"].value_counts().sort_index().items():
        plt.text(i, count + m.ceil(m.log(count,10))*10, str(count), ha="center")
    plt.show()
plot_cluster_distribution(data_labeled)

### Cluster Distribution Observations 
 

### Separate Data by Clusters

In [ ]:
df_labeled = label_data(df_clean, best_model)
clusters_df = model_breakdown(best_model, df_labeled)

## Final Propositions


## Acknowledgement of Contributions
### Group 3 Members:  
Sebastian Levy  
Idongesit Orok  
Miracle Okojie  

### Task Distribution:  


We, Group 3, acknowledge that we understand all parts of this assignment, including those which we did not personally carry out.